# K562 ATAC-seq data chr22

In [ ]:
import pysam

In [ ]:
def subsample_fasta_to_chr22(input_fasta, output_fasta, chromsizes_file):
    """
    Subsamples a FASTA file to include only chr22 and writes the corresponding chromsizes file.
    
    Parameters:
        input_fasta (str): Path to the input FASTA file.
        output_fasta (str): Path to the output FASTA file containing only chr22.
        chromsizes_file (str): Path to the output chromsizes file for chr22.
    """
    with pysam.FastaFile(input_fasta) as fasta_in, open(output_fasta, 'w') as fasta_out, open(chromsizes_file, 'w') as chromsizes_out:
        # Check if chr22 exists in the reference
        if "chr22" not in fasta_in.references:
            raise ValueError("chr22 not found in the input FASTA file.")
        
        # Get chr22 sequence and write it to the output FASTA
        chr22_sequence = fasta_in.fetch("chr22")
        chr22_length = fasta_in.get_reference_length("chr22")
        
        fasta_out.write(f">chr22\n")
        for i in range(0, len(chr22_sequence), 80):  # Wrap sequence at 80 characters
            fasta_out.write(chr22_sequence[i:i+80] + "\n")
        
        # Write chr22 chromsize to the chromsizes file
        chromsizes_out.write(f"chr22\t{chr22_length}\n")

In [ ]:
# Define input and output paths
input_fasta = "/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/hg38.fa"
output_fasta = "/cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/hg38.chr22.fa"
chromsizes_file = "/cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/hg38.chr22.chromsizes"

In [ ]:
# Run the subsampling function
subsample_fasta_to_chr22(input_fasta, output_fasta, chromsizes_file)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [ ]:
input_bed = "/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/ENCSR868FGK_K562_ATAC-seq_peaks.bed"
output_bed = "/cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22.bed"

In [ ]:
# Subsample the BED file to include only chr22
with open(input_bed, 'r') as bed_in, open(output_bed, 'w') as bed_out:
    for line in bed_in:
        if line.startswith("chr22"):
            bed_out.write(line)

In [ ]:
input_bam = '/cellar/users/aklie/data/datasets/SeqDatasets/K562_ATAC-seq/data/merged.bam'
output_bam = '/cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22.bam'

In [ ]:
# Subsample bam file to include only chr22 using pysam
def subsample_bam_to_chr22(input_bam, output_bam):
    """
    Subsamples a BAM file to include only reads mapped to chr22.
    
    Parameters:
        input_bam (str): Path to the input BAM file.
        output_bam (str): Path to the output BAM file with only chr22 reads.
    """
    # Open the input BAM file for reading
    with pysam.AlignmentFile(input_bam, "rb") as bam_in:
        # Open the output BAM file for writing
        with pysam.AlignmentFile(output_bam, "wb", header=bam_in.header) as bam_out:
            # Iterate over reads mapped to chr22 and write them to the output file
            for read in bam_in.fetch("chr22"):
                bam_out.write(read)

# Run the subsampling function
subsample_bam_to_chr22(input_bam, output_bam)

In [ ]:
# index the output bam file
pysam.index(output_bam)

''

%%bash
# use chrombpnet to get unstranded counts bigwig with correct shift (+4/-4)
source activate chrombpnet
script=/cellar/users/aklie/opt/chrombpnet/chrombpnet/helpers/preprocessing/reads_to_bigwig.py
cmd="python $script \
--genome /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/hg38.chr22.fa \
--input-bam-file /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22.bam \
--chrom-sizes /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/hg38.chr22.chromsizes \
--output-prefix /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22 \
--data-type ATAC"
echo $cmd
eval $cmd

%%bash
# use bam2bw to get bigwig
#bam2bw [-h] -s SIZES -n NAME [-ps POS_SHIFT] [-ns NEG_SHIFT] [-v] filename [filename ...]
cmd="bam2bw \
-s /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/hg38.chr22.chromsizes \
-n /cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22.bam2bw \
/cellar/users/aklie/projects/ML4GLand/SeqData/tests/data/K562_ATAC-seq_chr22/ENCSR868FGK.chr22.bam"
echo $cmd
eval $cmd